# 03 — Feature Engineering

## Objective

This notebook constructs model-ready features for forecasting HOEP.

Feature engineering is guided by the statistical analysis performed in the previous notebook, including ACF, PACF, and ADF tests.

Key insights:

- Strong short-term dependencies (lags 1–3)
- Daily seasonality (lag 24)
- Weekly seasonality (lag 168)
- Stationary series

Based on these findings, features are created differently for:

- statistical models
- machine learning models
- deep learning models

In [50]:
import pandas as pd
import numpy as np 
from pyspark.sql import SparkSession
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Load data 

In [51]:

spark = (
    SparkSession.builder
    .appName("HOEP_Feature_Engineering")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.parquet("hdfs://localhost:9000/hoep_project/processed/master_dataset.parquet")
pdf = df.toPandas()

pdf = pdf.sort_values(["date", "hour"]).reset_index(drop=True)

# Lag Features(based on our EDA file)




In [52]:
# Short-term lags
pdf["lag_1"] = pdf["hoep"]
pdf["lag_2"] = pdf["hoep"].shift(1)
pdf["lag_3"] = pdf["hoep"].shift(2)

# Daily seasonality
pdf["lag_24"] = pdf["hoep"].shift(23)

# Weekly seasonality
pdf["lag_168"] = pdf["hoep"].shift(167)

# Rolling Features

In time series forecasting, machine learning models such as linear regression, random forests, and gradient boosting do not inherently understand temporal dependencies. Unlike sequential models (e.g., LSTM), these models treat each observation as independent and rely entirely on the provided features to learn patterns.

While lag features (e.g., lag 1, lag 2, lag 3) provide direct access to recent past values, they only capture very short-term information and may not be sufficient to represent broader temporal dynamics such as trends or volatility.

To address this limitation, rolling features are introduced.

### What are Rolling Features?

Rolling features summarize recent historical behavior over a fixed window. For example:

- `rolling_mean_24`: average HOEP over the past 24 hours  
- `rolling_std_24`: standard deviation over the past 24 hours  

These features provide a smoothed representation of the time series.

### Why Rolling Features are Important

Rolling features help machine learning models by:

- **Capturing local trends:**  
  They indicate whether the current value is high or low relative to recent history.

- **Providing context:**  
  The same price can have different meanings depending on recent behavior. Rolling statistics allow the model to distinguish these situations.

- **Reducing noise:**  
  Electricity prices are highly volatile. Rolling averages smooth short-term fluctuations and help stabilize model learning.

- **Encoding temporal structure:**  
  Since traditional ML models do not have memory, rolling features act as a way to inject information about recent patterns into the model.

### Example Intuition

Consider two situations where the current price is the same:

- Scenario A: prices have been stable around this value  
- Scenario B: prices have recently dropped from much higher levels  

Without rolling features, both scenarios look identical to the model.  
With rolling features, the model can distinguish between them using differences in rolling mean and variance.

These features complement lag variables and improve the ability of machine learning models to capture both short-term dynamics and seasonal patterns in electricity prices.

> Note: Rolling features are primarily used for machine learning models. Deep learning models (e.g., LSTM) learn temporal patterns directly from sequences and do not require manually engineered rolling statistics.

In [53]:
pdf["rolling_mean_24"] = pdf["hoep"].rolling(24).mean()
pdf["rolling_std_24"] = pdf["hoep"].rolling(24).std()
pdf["rolling_mean_168"] = pdf["hoep"].rolling(168).mean()

## Horizon-Specific Feature Alignment (For ML Models)

To avoid data leakage and ensure realistic forecasting, features are aligned according to whether they are known at prediction time.

- Lagged price variables and exogenous variables such as demand and generation are taken from the current or past time steps, since these values are available when generating a forecast.
- Calendar variables such as hour of day, day of week, and weekend indicators are aligned with the forecast horizon, since they are known in advance.

For example, when predicting the price at time \(t+1\):

- lag and exogenous variables come from time \(t\) or earlier
- calendar variables correspond to time \(t+1\)

This logic is extended separately for 2-step and 3-step ahead forecasting.

In [54]:
# For Horizon 1 
pdf["hour_t_plus_1"] = pdf["hour"].shift(-1)
pdf["day_of_week_t_plus_1"] = pdf["day_of_week"].shift(-1)
pdf["is_weekend_t_plus_1"] = pdf["is_weekend"].shift(-1)
pdf["month_t_plus_1"] = pdf["month"].shift(-1)
pdf["season_t_plus_1"] = pdf["season"].shift(-1)

# For Horizon 2
pdf["hour_t_plus_2"] = pdf["hour"].shift(-2)
pdf["day_of_week_t_plus_2"] = pdf["day_of_week"].shift(-2)
pdf["is_weekend_t_plus_2"] = pdf["is_weekend"].shift(-2)
pdf["month_t_plus_2"] = pdf["month"].shift(-2)
pdf["season_t_plus_2"] = pdf["season"].shift(-2)

# For Horizon 1 
pdf["hour_t_plus_3"] = pdf["hour"].shift(-3)
pdf["day_of_week_t_plus_3"] = pdf["day_of_week"].shift(-3)
pdf["is_weekend_t_plus_3"] = pdf["is_weekend"].shift(-3)
pdf["month_t_plus_3"] = pdf["month"].shift(-3)
pdf["season_t_plus_3"] = pdf["season"].shift(-3)

# Data Splitting Strategy

To preserve the temporal structure of the series, the dataset is split chronologically into training, validation, and test sets.

- The **training set** includes observations from January 2023 to December 2024.
- The **validation set** includes observations from January 2025 to March 2025 and is used for hyperparameter tuning and model selection.
- The **test set** consists of the final four weeks of available data in April 2025.

In addition to reporting overall test performance, the test period is further divided into weekly segments(in evaluation section). This allows model performance to be evaluated across different weeks within the final month of data. Such an analysis provides insight into the temporal stability of forecasting performance and may indicate whether predictive accuracy degrades as the evaluation period moves farther away from the training period.

This weekly evaluation is also useful from a practical perspective, as it provides an initial indication of whether model monitoring and periodic retraining may be necessary in a real-world forecasting setting.

In [55]:
# Define split boundaries
train_end = pd.Timestamp("2024-12-31")
val_start = pd.Timestamp("2025-01-01")
val_end = pd.Timestamp("2025-03-31")
test_start = pd.Timestamp("2025-04-01")
test_end = pd.Timestamp("2025-04-30")

# Creat split
train_df = pdf[pdf["date"] <= train_end]

val_df = pdf[
    (pdf["date"] >= val_start) &
    (pdf["date"] <= val_end)
]

test_df = pdf[
    (pdf["date"] >= test_start) &
    (pdf["date"] <= test_end)
]

# Sanity check

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain range:", train_df["date"].min(), "→", train_df["date"].max())
print("Validation range:", val_df["date"].min(), "→", val_df["date"].max())
print("Test range:", test_df["date"].min(), "→", test_df["date"].max())

Train: (17544, 46)
Validation: (2160, 46)
Test: (720, 46)

Train range: 2023-01-01 00:00:00 → 2024-12-31 00:00:00
Validation range: 2025-01-01 00:00:00 → 2025-03-31 00:00:00
Test range: 2025-04-01 00:00:00 → 2025-04-30 00:00:00


# Model-Specific Dataset Preparation

After constructing the engineered features and splitting the data into training, validation, and test sets, separate modeling datasets are created for each class of forecasting models.

This step is necessary because different model families have different input requirements.

### 1. Statistical Models

Statistical forecasting models such as ARIMA and SARIMA operate directly on the time series structure and typically rely on the target series itself, with optional seasonal or exogenous components.  
For this reason, the statistical modeling dataset primarily consists of the chronologically ordered HOEP series, along with optional exogenous variables when required.

### 2. Machine Learning Models

Machine learning models such as Random Forest and XGBoost require a tabular feature matrix.  
Therefore, the dataset for these models includes:

- lag features
- rolling statistics
- calendar variables
- exogenous variables such as demand and generation

These models do not automatically learn temporal structure, so the relevant time-dependent information must be explicitly provided through engineered features.

### 3. Deep Learning Models

Deep learning models such as LSTM and GRU require sequential inputs rather than independent feature rows.  
For this reason, the deep learning dataset is built by transforming the tabular data into fixed-length input sequences. Each sequence represents a rolling window of historical observations, and the target corresponds to the future value to be predicted.

In this project, separate datasets are created for:

- statistical models
- machine learning models
- deep learning models

This design ensures that each modeling approach receives data in the format most appropriate for its underlying learning mechanism.

In [56]:
# 1. Define target and exogenous features:
target_col = "hoep"

features_exog = [
    "market_demand",
    "ontario_demand",
    "nuclear",
    "gas",
    "hydro",
    "wind",
    "solar",
    "biofuel"
]

In [57]:
# 2. Define ML feature columns 
# for horizon 1 
feature_cols_ml_h1 = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
    "hour_t_plus_1",
    "day_of_week_t_plus_1",
    "is_weekend_t_plus_1",
    "month_t_plus_1",
    "season_t_plus_1"
] + features_exog

# for horizon 2
feature_cols_ml_h2 = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
    "hour_t_plus_2",
    "day_of_week_t_plus_2",
    "is_weekend_t_plus_2",
    "month_t_plus_2",
    "season_t_plus_2"
] + features_exog

#for horizon 3
feature_cols_ml_h3 = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
    "hour_t_plus_3",
    "day_of_week_t_plus_3",
    "is_weekend_t_plus_3",
    "month_t_plus_3",
    "season_t_plus_3"
] + features_exog

In [58]:
# 3. define DL feature columns

feature_cols_dl = [
    "hoep"
] + features_exog

### 1. Statistical  Modeling Dataset 

In [59]:
stat_train = train_df[["date", "hour", target_col] + features_exog].copy()
stat_val = val_df[["date", "hour", target_col] + features_exog].copy()
stat_test = test_df[["date", "hour", target_col] + features_exog].copy()

### 2. Machine learning modeling dataset

In [60]:
def make_direct_dataset(df, feature_cols, target_col, horizon):
    X = df[feature_cols].iloc[:-horizon].copy()
    y = df[target_col].shift(-horizon).iloc[:-horizon].copy()
    return X, y

# for horizon 1
X_train_ml_h1, y_train_ml_h1 = make_direct_dataset(train_df, feature_cols_ml_h1, target_col, 1)
X_val_ml_h1, y_val_ml_h1 = make_direct_dataset(val_df, feature_cols_ml_h1, target_col, 1)
X_test_ml_h1, y_test_ml_h1 = make_direct_dataset(test_df, feature_cols_ml_h1, target_col, 1)

# for horizon 2
X_train_ml_h2, y_train_ml_h2 = make_direct_dataset(train_df, feature_cols_ml_h2, target_col, 2)
X_val_ml_h2, y_val_ml_h2 = make_direct_dataset(val_df, feature_cols_ml_h2, target_col, 2)
X_test_ml_h2, y_test_ml_h2 = make_direct_dataset(test_df, feature_cols_ml_h2, target_col, 2)

# for horizon 3 
X_train_ml_h3, y_train_ml_h3 = make_direct_dataset(train_df, feature_cols_ml_h3, target_col, 3)
X_val_ml_h3, y_val_ml_h3 = make_direct_dataset(val_df, feature_cols_ml_h3, target_col, 3)
X_test_ml_h3, y_test_ml_h3 = make_direct_dataset(test_df, feature_cols_ml_h3, target_col, 3)



### 3. Deep Learning modeling dataset

In [61]:
# Scaling the features using standardscaler()

scaler_dl = StandardScaler()

X_train_dl_scaled = scaler_dl.fit_transform(train_df[feature_cols_dl])
X_val_dl_scaled = scaler_dl.transform(val_df[feature_cols_dl])
X_test_dl_scaled = scaler_dl.transform(test_df[feature_cols_dl])

y_train_dl = train_df[target_col].values # because pytorch works with numpy arrays
y_val_dl = val_df[target_col].values
y_test_dl = test_df[target_col].values

In [62]:
# Ceeate Deep learning sequence
# Based on our earlier reasoning ( EDA file ) we choose 168 hours (weeekly window)
def create_sequences(X: pd.DataFrame, y: pd.DataFrame,
                     seq_length: int=168, horizon: int=1) -> tuple[np.ndarray, np.ndarray]:
    X_seq, y_seq = [], []

    for i in range(len(X)-seq_length-horizon+1):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i + seq_length + horizon - 1])

    return np.array(X_seq), np.array(y_seq)

In [63]:
# 1 step ahead 
X_train_dl_1, y_train_dl_1 = create_sequences(X_train_dl_scaled, y_train_dl, seq_length=168, horizon=1)
X_val_dl_1, y_val_dl_1 = create_sequences(X_val_dl_scaled, y_val_dl, seq_length=168, horizon=1)
X_test_dl_1, y_test_dl_1 = create_sequences(X_test_dl_scaled, y_test_dl, seq_length=168, horizon=1)

In [64]:
# 2 step ahead 
X_train_dl_2, y_train_dl_2 = create_sequences(X_train_dl_scaled, y_train_dl, seq_length=168, horizon=2)
X_val_dl_2, y_val_dl_2 = create_sequences(X_val_dl_scaled, y_val_dl, seq_length=168, horizon=2)
X_test_dl_2, y_test_dl_2 = create_sequences(X_test_dl_scaled, y_test_dl, seq_length=168, horizon=2)

In [65]:
# 3 step ahead
X_train_dl_3, y_train_dl_3 = create_sequences(X_train_dl_scaled, y_train_dl, seq_length=168, horizon=3)
X_val_dl_3, y_val_dl_3 = create_sequences(X_val_dl_scaled, y_val_dl, seq_length=168, horizon=3)
X_test_dl_3, y_test_dl_3 = create_sequences(X_test_dl_scaled, y_test_dl, seq_length=168, horizon=3)

# Saving Model-Specific Datasets

Because statistical, machine learning, and deep learning models require different input structures, the processed datasets are saved separately by model type.

- **Statistical models** use chronologically ordered time series data.
- **Machine learning models** use tabular feature matrices.
- **Deep learning models** use sequence-based arrays.

Saving these datasets in separate folders improves organization, reproducibility, and ease of loading in later modeling notebooks.

In [66]:
PROJECT_ROOT = Path.cwd().parent

stat_dir =PROJECT_ROOT/'data'/'processed'/'stat_dir'
ml_dir =PROJECT_ROOT/'data'/'processed'/'ml_dir'
dl_dir =PROJECT_ROOT/'data'/'processed'/'dl_dir'

for path in [stat_dir, ml_dir, dl_dir]:
    path.mkdir(parents=True, exist_ok=True)




In [67]:
# Save statistical model datasets
stat_train.to_parquet(stat_dir / "train.parquet", index=False)
stat_val.to_parquet(stat_dir / "val.parquet", index=False)
stat_test.to_parquet(stat_dir / "test.parquet", index=False)

In [ ]:
# Save machine learning model datasets
datasets = {
    1: (X_train_ml_h1, X_val_ml_h1, X_test_ml_h1, y_train_ml_h1, y_val_ml_h1, y_test_ml_h1),
    2: (X_train_ml_h2, X_val_ml_h2, X_test_ml_h2, y_train_ml_h2, y_val_ml_h2, y_test_ml_h2),
    3: (X_train_ml_h3, X_val_ml_h3, X_test_ml_h3, y_train_ml_h3, y_val_ml_h3, y_test_ml_h3),
}

for i, (X_train, X_val, X_test, y_train, y_val, y_test) in datasets.items():
    X_train.to_parquet(ml_dir / f"X_train_h{i}.parquet", index=False)
    X_val.to_parquet(ml_dir / f"X_val_h{i}.parquet", index=False)
    X_test.to_parquet(ml_dir / f"X_test_h{i}.parquet", index=False)
     
    y_train.to_frame().to_parquet(ml_dir / f"y_train_h{i}.parquet", index=False)
    y_val.to_frame().to_parquet(ml_dir / f"y_val_h{i}.parquet", index=False)
    y_test.to_frame().to_parquet(ml_dir / f"y_test_h{i}.parquet", index=False)

In [69]:
# save deep learning models dataset
np.save(dl_dir / "X_train_h1.npy", X_train_dl_1)
np.save(dl_dir / "y_train_h1.npy", y_train_dl_1)
np.save(dl_dir / "X_val_h1.npy", X_val_dl_1)
np.save(dl_dir / "y_val_h1.npy", y_val_dl_1)
np.save(dl_dir / "X_test_h1.npy", X_test_dl_1)
np.save(dl_dir / "y_test_h1.npy", y_test_dl_1)

np.save(dl_dir / "X_train_h2.npy", X_train_dl_2)
np.save(dl_dir / "y_train_h2.npy", y_train_dl_2)
np.save(dl_dir / "X_val_h2.npy", X_val_dl_2)
np.save(dl_dir / "y_val_h2.npy", y_val_dl_2)
np.save(dl_dir / "X_test_h2.npy", X_test_dl_2)
np.save(dl_dir / "y_test_h2.npy", y_test_dl_2)

np.save(dl_dir / "X_train_h3.npy", X_train_dl_3)
np.save(dl_dir / "y_train_h3.npy", y_train_dl_3)
np.save(dl_dir / "X_val_h3.npy", X_val_dl_3)
np.save(dl_dir / "y_val_h3.npy", y_val_dl_3)
np.save(dl_dir / "X_test_h3.npy", X_test_dl_3)
np.save(dl_dir / "y_test_h3.npy", y_test_dl_3)